# 10 · Treino — Risco de OLA como Regressão (taxa de violação, D+1 / D+7)

Mesmo padrão do notebook `08` (Pipeline + `GBTRegressor` + portão
automático vs. baseline), mas prevendo **taxa de violação de KPI**
(`target_taxa_d1`/`target_taxa_d7`) em vez de volume de incidentes.

| Tabela | Segmentação categórica | Horizontes |
|---|---|---|
| `features_risco_ola_produto` | `produto` | D+1, D+7 |
| `features_risco_ola_equipe` | `grupo_designado` | D+1, D+7 |

**Baseline**: `taxa_media_movel_7d` (já existe na tabela — "a taxa de
violação vai continuar parecida com a média recente").

**Features de contexto incluídas**: `qtd_kpi_regra_divergente` e
`qtd_duracao_suspeita` (achados do notebook `04`/`05`) — sinal de
qualidade de dado, não usados pra "corrigir" o alvo, só como informação
a mais pro modelo.

Sem MLflow nesta versão (mesma decisão do `08`).

In [ ]:
%run ./00_config

In [ ]:
%run ./07_ml_prep

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator


def _mae_rmse(df, coluna_target: str, coluna_previsao: str):
    ev_mae = RegressionEvaluator(labelCol=coluna_target, predictionCol=coluna_previsao, metricName="mae")
    ev_rmse = RegressionEvaluator(labelCol=coluna_target, predictionCol=coluna_previsao, metricName="rmse")
    return ev_mae.evaluate(df), ev_rmse.evaluate(df)


print("_mae_rmse() pronta.")

## Função de treino com portão automático — risco de OLA

Mesma estrutura do `treinar_com_gate` do `08` (split antes do
`dropna`, `Pipeline` unificado, busca pequena de hiperparâmetro), só
trocando as colunas de lag/baseline pelas equivalentes de taxa.

In [ ]:
CANDIDATOS_HIPERPARAMETRO_RISCO = [
    {"maxDepth": 3, "maxIter": 30},
    {"maxDepth": 5, "maxIter": 50},
    {"maxDepth": 7, "maxIter": 80},
]


def treinar_risco_com_gate(nome_tabela: str, coluna_target: str, coluna_segmento_categorica: str = None) -> dict:
    """
    Treina até 3 candidatos de GBTRegressor (dentro de um Pipeline com
    StringIndexer+VectorAssembler) para prever taxa de violação de KPI,
    escolhe o melhor pela validação, e só declara "vencedor" se bater a
    baseline (taxa_media_movel_7d) na validação.
    """
    colunas_lag = ["taxa_lag_1d", "taxa_lag_7d", "taxa_media_movel_7d", "taxa_media_movel_14d"]

    df = carregar_features_com_calendario(nome_tabela).withColumn(
        "is_fim_de_semana_int", F.col("is_fim_de_semana").cast("int")
    )

    treino_bruto, validacao_bruto, teste_bruto = split_temporal(df)
    treino = treino_bruto.dropna(subset=colunas_lag + [coluna_target])
    validacao = validacao_bruto.dropna(subset=colunas_lag + [coluna_target])
    teste = teste_bruto.dropna(subset=colunas_lag + [coluna_target])

    colunas_numericas = list(colunas_lag) + [
        "dia_semana_num", "trimestre", "is_feriado",
        "qtd_kpi_regra_divergente", "qtd_duracao_suspeita", "is_fim_de_semana_int",
    ]
    if coluna_segmento_categorica != "prioridade_num":
        colunas_numericas.append("prioridade_num")

    stages = []
    colunas_features = list(colunas_numericas)
    n_categorias = 0
    if coluna_segmento_categorica:
        indexer = StringIndexer(
            inputCol=coluna_segmento_categorica,
            outputCol=f"{coluna_segmento_categorica}_idx",
            handleInvalid="keep",
        )
        stages.append(indexer)
        colunas_features.append(f"{coluna_segmento_categorica}_idx")
        n_categorias = df.select(coluna_segmento_categorica).distinct().count()

    assembler = VectorAssembler(inputCols=colunas_features, outputCol="features")
    stages.append(assembler)
    max_bins = max(32, n_categorias + 5)

    mae_baseline_val, _ = _mae_rmse(validacao, coluna_target, "taxa_media_movel_7d")

    melhor = None
    for cand in CANDIDATOS_HIPERPARAMETRO_RISCO:
        gbt = GBTRegressor(
            featuresCol="features", labelCol=coluna_target,
            maxIter=cand["maxIter"], maxDepth=cand["maxDepth"], maxBins=max_bins, seed=42,
        )
        pipeline = Pipeline(stages=stages + [gbt])
        pipeline_ajustado = pipeline.fit(treino)
        mae_val, rmse_val = _mae_rmse(pipeline_ajustado.transform(validacao), coluna_target, "prediction")
        if melhor is None or mae_val < melhor["mae_val"]:
            melhor = {**cand, "mae_val": mae_val, "rmse_val": rmse_val, "pipeline": pipeline_ajustado}

    vencedor = "modelo" if melhor["mae_val"] < mae_baseline_val else "baseline"

    if vencedor == "modelo":
        mae_teste, rmse_teste = _mae_rmse(melhor["pipeline"].transform(teste), coluna_target, "prediction")
    else:
        mae_teste, rmse_teste = _mae_rmse(teste, coluna_target, "taxa_media_movel_7d")

    return {
        "vencedor": vencedor,
        "hiperparametros": f"maxDepth={melhor['maxDepth']},maxIter={melhor['maxIter']}" if vencedor == "modelo" else "-",
        "mae_val_modelo": round(melhor["mae_val"], 4),
        "mae_val_baseline": round(mae_baseline_val, 4),
        "mae_teste_final": round(mae_teste, 4),
        "rmse_teste_final": round(rmse_teste, 4),
        "n_treino": treino.count(),
        "n_validacao": validacao.count(),
        "n_teste": teste.count(),
        "pipeline_ajustado": melhor["pipeline"] if vencedor == "modelo" else None,
        "teste_bruto": teste_bruto,
        "coluna_segmento_categorica": coluna_segmento_categorica,
    }


print("treinar_risco_com_gate() pronta.")

## Treinar as 4 combinações

In [ ]:
configuracoes_risco = [
    ("features_risco_ola_produto", "produto"),
    ("features_risco_ola_equipe", "grupo_designado"),
]
horizontes_risco = ["target_taxa_d1", "target_taxa_d7"]

resultados_risco_por_chave = {}
resumo_resultados_risco = []

for nome_tabela, coluna_segmento in configuracoes_risco:
    for horizonte in horizontes_risco:
        chave = f"{nome_tabela}__{horizonte}"
        print(f"Treinando: {chave}")
        resultado = treinar_risco_com_gate(nome_tabela, horizonte, coluna_segmento)
        resultados_risco_por_chave[chave] = resultado
        resumo_resultados_risco.append({
            "tabela": nome_tabela, "horizonte": horizonte, "vencedor": resultado["vencedor"],
            "hiperparametros": resultado["hiperparametros"],
            "mae_val_modelo": resultado["mae_val_modelo"], "mae_val_baseline": resultado["mae_val_baseline"],
            "mae_teste_final": resultado["mae_teste_final"], "rmse_teste_final": resultado["rmse_teste_final"],
        })

print(f"\n{len(resultados_risco_por_chave)} combinações avaliadas.")

## Resumo comparativo

⚠️ Mesmo alerta do notebook `08`: segmentos com poucas linhas de
validação podem ter um "vencedor modelo" que não generaliza tão bem
quanto a validação sugere — reportar com essa ressalva, não como
certeza absoluta.

In [ ]:
resumo_risco_df = spark.createDataFrame(resumo_resultados_risco)
display(resumo_risco_df.select(
    "tabela", "horizonte", "vencedor", "hiperparametros",
    "mae_val_modelo", "mae_val_baseline", "mae_teste_final", "rmse_teste_final",
))

qtd_modelo = sum(1 for r in resumo_resultados_risco if r["vencedor"] == "modelo")
qtd_baseline = sum(1 for r in resumo_resultados_risco if r["vencedor"] == "baseline")
print(f"\nModelo venceu em {qtd_modelo} de {len(resumo_resultados_risco)} combinações.")
print(f"Baseline venceu em {qtd_baseline} de {len(resumo_resultados_risco)} combinações.")